# 23b_evaluate_ensemble — 표현앙상블 모델 평가 6종 (신규)

**한 줄 요약:** 배포 모델(표현앙상블)을 **Learning Curve·ROC/AUC·Confusion Matrix·Prediction Error·Calibration·Feature Importance** 6가지로 검증한다.
**평가 대상:** 22의 재균형 test셋(active 308·decoy 30·real_inactive 30).
**출력:** `data/ensemble_evaluation.png` (6분할 그림).
> 각 그래프의 의미·해석법은 대화 설명 참고.

### 준비 + 전체 평가 실행
모델·데이터를 불러와 6개 평가 그래프를 한 번에 그린다.

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import os, numpy as np, pandas as pd, pickle
from collections import defaultdict
from sklearn.base import clone
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix,
                             matthews_corrcoef, accuracy_score)
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- 모델·데이터 로드 ---
with open("data/HSD17B13_repr_ensemble.pkl","rb") as f: B=pickle.load(f)
TOP, REPS, PIPES = B["top"], B["reps"], B["pipelines"]
feat = pd.read_csv("data/HSD17B13_final_training_1to1_v2.csv")
mem  = pd.read_csv("data/HSD17B13_rebalanced_membership.csv")
df = mem.merge(feat.drop(columns=["potency"]), on="canonical_smiles", how="left")

def Xy(rep, split):
    m = df["split"]==split
    return df.loc[m, REPS[rep]].to_numpy(), df.loc[m,"potency"].to_numpy(), df.loc[m,"source"].to_numpy()

def ens_proba_from(fitted, split):     # fitted: dict key->pipeline
    ps=[]
    for mn,rn in TOP:
        X,_,_ = Xy(rn, split)
        ps.append(fitted[f"{mn}|{rn}"].predict_proba(X)[:,1])
    return np.mean(ps,axis=0)

ytr = df.loc[df.split=="train","potency"].to_numpy()
yte = df.loc[df.split=="test","potency"].to_numpy()
src_te = df.loc[df.split=="test","source"].to_numpy()
p_te = ens_proba_from(PIPES, "test")
pred_te = (p_te>=0.5).astype(int)
print("test MCC %.3f | ROC-AUC %.3f | Acc %.3f" %
      (matthews_corrcoef(yte,pred_te), roc_auc_score(yte,p_te), accuracy_score(yte,pred_te)))

fig, ax = plt.subplots(3,2, figsize=(13,15)); fig.suptitle("HSD17B13 Representation-Ensemble — Evaluation", fontsize=15, y=0.995)

# 1) Learning Curve (accuracy: train subset vs validation)
sizes=[0.2,0.4,0.6,0.8,1.0]; tr_idx=df.index[df.split=="train"]
tr_sc, va_sc = [], []
for f in sizes:
    sub = df.loc[tr_idx].sample(frac=f, random_state=1).index
    fitted={}
    for mn,rn in TOP:
        pipe=clone(PIPES[f"{mn}|{rn}"])
        Xs=df.loc[sub, REPS[rn]].to_numpy(); ys=df.loc[sub,"potency"].to_numpy()
        fitted[f"{mn}|{rn}"]=pipe.fit(Xs,ys)
    # train score on subset
    ps=[fitted[f"{mn}|{rn}"].predict_proba(df.loc[sub,REPS[rn]].to_numpy())[:,1] for mn,rn in TOP]
    tr_sc.append(accuracy_score(df.loc[sub,"potency"], (np.mean(ps,0)>=0.5).astype(int)))
    va_sc.append(accuracy_score(df.loc[df.split=="val","potency"], (ens_proba_from(fitted,"val")>=0.5).astype(int)))
n_tr=[int(len(tr_idx)*f) for f in sizes]
ax[0,0].plot(n_tr,tr_sc,"o-",label="Training score")
ax[0,0].plot(n_tr,va_sc,"s-",label="Validation score",color="green")
ax[0,0].set_title("1) Learning Curve"); ax[0,0].set_xlabel("Training samples"); ax[0,0].set_ylabel("Accuracy")
ax[0,0].legend(); ax[0,0].grid(alpha=.3)

# 2) ROC / AUC
fpr,tpr,_=roc_curve(yte,p_te); auc=roc_auc_score(yte,p_te)
ax[0,1].plot(fpr,tpr,label=f"AUC = {auc:.3f}"); ax[0,1].plot([0,1],[0,1],"k--",alpha=.5)
ax[0,1].set_title("2) ROC Curve"); ax[0,1].set_xlabel("False Positive Rate"); ax[0,1].set_ylabel("True Positive Rate")
ax[0,1].legend(loc="lower right"); ax[0,1].grid(alpha=.3)

# 3) Confusion Matrix
cm=confusion_matrix(yte,pred_te,labels=[0,1])
im=ax[1,0].imshow(cm,cmap="Greens")
for (i,j),v in np.ndenumerate(cm): ax[1,0].text(j,i,str(v),ha="center",va="center",
    color="white" if v>cm.max()/2 else "black",fontsize=14)
ax[1,0].set_xticks([0,1]); ax[1,0].set_xticklabels(["inactive(0)","active(1)"])
ax[1,0].set_yticks([0,1]); ax[1,0].set_yticklabels(["inactive(0)","active(1)"])
ax[1,0].set_title("3) Confusion Matrix"); ax[1,0].set_xlabel("Predicted"); ax[1,0].set_ylabel("Actual")

# 4) Prediction Error by source (stacked: actual group -> predicted class)
groups=["active","decoy","real_inactive"]
pred1=np.array([ (pred_te[src_te==g]==1).sum() for g in groups])
pred0=np.array([ (pred_te[src_te==g]==0).sum() for g in groups])
ax[1,1].bar(groups,pred0,label="pred inactive(0)",color="#8ecae6")
ax[1,1].bar(groups,pred1,bottom=pred0,label="pred active(1)",color="#fb8500")
for i,g in enumerate(groups): ax[1,1].text(i,(pred0[i]+pred1[i])+1,f"n={pred0[i]+pred1[i]}",ha="center",fontsize=9)
ax[1,1].set_title("4) Class Prediction Error (by source)"); ax[1,1].set_ylabel("count"); ax[1,1].legend()

# 5) Calibration curve
frac,mean_pred=calibration_curve(yte,p_te,n_bins=10,strategy="uniform")
ax[2,0].plot(mean_pred,frac,"o-",label="Ensemble"); ax[2,0].plot([0,1],[0,1],"k--",alpha=.5,label="Perfect")
ax[2,0].set_title("5) Calibration Curve"); ax[2,0].set_xlabel("Mean predicted prob"); ax[2,0].set_ylabel("Fraction of positives")
ax[2,0].legend(); ax[2,0].grid(alpha=.3)

# 6) Feature Importance (aggregate over tree bases)
imp=defaultdict(float)
for mn,rn in TOP:
    model=PIPES[f"{mn}|{rn}"].named_steps["m"]
    fi=getattr(model,"feature_importances_",None)
    if fi is None: continue
    for c,val in zip(REPS[rn],fi):
        name = c if rn=="desc2d" else f"{rn}:{c.split('_')[-1]}"
        imp[name]+=val/len(TOP)
topf=sorted(imp.items(),key=lambda x:x[1],reverse=True)[:15][::-1]
ax[2,1].barh([k for k,_ in topf],[v for _,v in topf],color="#219ebc")
ax[2,1].set_title("6) Feature Importance (ensemble aggregate, top15)"); ax[2,1].set_xlabel("mean importance")
ax[2,1].tick_params(axis="y",labelsize=8)

plt.tight_layout(rect=[0,0,1,0.99])
plt.savefig("data/ensemble_evaluation.png",dpi=130,bbox_inches="tight")
print("저장: data/ensemble_evaluation.png")

# 콘솔용 수치 요약
tn,fp,fn,tp=cm.ravel()
print(f"혼동행렬: TP={tp} TN={tn} FP={fp} FN={fn}")
print(f"source별 정답률: active {(pred_te[src_te=='active']==1).mean():.2f} | "
      f"decoy {(pred_te[src_te=='decoy']==0).mean():.2f} | real_inactive {(pred_te[src_te=='real_inactive']==0).mean():.2f}")
print("상위 feature 5:", [k for k,_ in topf[::-1][:5]])
